## 1. Load Dataset



In [ ]:
import pandas as pd

df = pd.read_csv('/content/sample_data/story_qa_dataset.csv')

df


,story,question,answer
0,Mina found a small blue umbrella near the scho...,Where did Mina find the umbrella?,Near the school gate.
1,Mina found a small blue umbrella near the scho...,What did Mina do with the umbrella?,She took it to the lost-and-found room.
2,Mina found a small blue umbrella near the scho...,Who came looking for the umbrella?,A younger student.
3,Rafi planted three tomato seeds in a pot. He p...,How many tomato seeds did Rafi plant?,Three.
4,Rafi planted three tomato seeds in a pot. He p...,Where did Rafi place the pot?,Beside a sunny window.
5,Rafi planted three tomato seeds in a pot. He p...,When did Rafi see the first green leaves?,After two weeks.
6,Nila was walking home when she noticed that th...,What did Nila buy?,One bun.
7,Nila was walking home when she noticed that th...,Who received half of the bun?,Her little brother.
8,Nila was walking home when she noticed that th...,Why did Nila buy only one bun?,She had only enough money for one bun.
9,Arman built a paper boat with his sister. They...,Who built the paper boat with Arman?,His sister.


## 2. Explore One Example




In [ ]:
index = 0

print("Story:")
print(df.iloc[index]['story'])

print("\nQuestion:")
print(df.iloc[index]['question'])

print("\nAnswer:")
print(df.iloc[index]['answer'])


Story:
Mina found a small blue umbrella near the school gate. She took it to the lost-and-found room. The next day, a younger student came looking for it.

Question:
Where did Mina find the umbrella?

Answer:
Near the school gate.


## 3. Tokenization


- lowercase
- quotes remove
- punctuation remove
- extra spaces remove
- `split()` দিয়ে words আলাদা


In [ ]:
import re

def tokenize(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Remove quotes
    text = re.sub(r"[\"']", "", text)

    # 3. Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # 4. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # 5. Split into words
    tokens = text.split()

    return tokens

print(tokenize(df['question'][0]))


['where', 'did', 'mina', 'find', 'the', 'umbrella']


## 4. Build Vocabulary



In [ ]:
vocab = {'<UNK>': 0}

def build_vocab(row):

    # Tokenize story, question and answer
    story_tokens = tokenize(row['story'])
    question_tokens = tokenize(row['question'])
    answer_tokens = tokenize(row['answer'])

    # Merge all tokens
    merged_tokens = story_tokens + question_tokens + answer_tokens

    # Add unique tokens
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

# Apply to every row
df.apply(build_vocab, axis=1)

print("Vocabulary size:", len(vocab))
print(vocab)


Vocabulary size: 159
{'<UNK>': 0, 'mina': 1, 'found': 2, 'a': 3, 'small': 4, 'blue': 5, 'umbrella': 6, 'near': 7, 'the': 8, 'school': 9, 'gate': 10, 'she': 11, 'took': 12, 'it': 13, 'to': 14, 'lost': 15, 'and': 16, 'room': 17, 'next': 18, 'day': 19, 'younger': 20, 'student': 21, 'came': 22, 'looking': 23, 'for': 24, 'where': 25, 'did': 26, 'find': 27, 'what': 28, 'do': 29, 'with': 30, 'who': 31, 'rafi': 32, 'planted': 33, 'three': 34, 'tomato': 35, 'seeds': 36, 'in': 37, 'pot': 38, 'he': 39, 'placed': 40, 'beside': 41, 'sunny': 42, 'window': 43, 'watered': 44, 'every': 45, 'morning': 46, 'after': 47, 'two': 48, 'weeks': 49, 'saw': 50, 'first': 51, 'green': 52, 'leaves': 53, 'how': 54, 'many': 55, 'plant': 56, 'place': 57, 'when': 58, 'see': 59, 'nila': 60, 'was': 61, 'walking': 62, 'home': 63, 'noticed': 64, 'that': 65, 'bakery': 66, 'still': 67, 'open': 68, 'had': 69, 'only': 70, 'enough': 71, 'money': 72, 'one': 73, 'bun': 74, 'so': 75, 'bought': 76, 'shared': 77, 'half': 78, 'her': 

## 5. Text to Indices



In [ ]:
def text_to_indices(text, vocab):

    indexed_text = []

    tokens = tokenize(text)

    for token in tokens:

        if token in vocab:
            indexed_text.append(vocab[token])

        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text


example = "Where did Mina find the umbrella?"

print(text_to_indices(example, vocab))


[25, 26, 1, 27, 8, 6]


## 6. PyTorch Dataset

প্রতিটি sample-এ:

**Input = Story + Question**

**Target = Answer**

এই simple version-এ পরে answer-এর first token target হিসেবে ব্যবহার করা হবে।


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):

    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):

        row = self.df.iloc[index]

        # Convert story to indices
        story_indices = text_to_indices(
            row['story'],
            self.vocab
        )

        # Convert question to indices
        question_indices = text_to_indices(
            row['question'],
            self.vocab
        )

        # Convert answer to indices
        answer_indices = text_to_indices(
            row['answer'],
            self.vocab
        )

        # Combine story + question
        input_indices = story_indices + question_indices

        return (
            torch.tensor(input_indices, dtype=torch.long),
            torch.tensor(answer_indices, dtype=torch.long)
        )


dataset = QADataset(df, vocab)

print("Number of samples:", len(dataset))
print(dataset[0])


Number of samples: 21
(tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  8, 15, 16,  2,
        17,  8, 18, 19,  3, 20, 21, 22, 23, 24, 13, 25, 26,  1, 27,  8,  6]), tensor([ 7,  8,  9, 10]))


## 7. DataLoader

`batch_size=1` ব্যবহার করছি কারণ প্রতিটি story/question-এর sequence length একই নয়।

যদি batch size > 1 করা হয়, তাহলে সাধারণভাবে padding/collate function দরকার হবে।


In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

for question, answer in dataloader:

    print("Input shape:", question.shape)
    print("Answer shape:", answer.shape)

    print("Input:", question)
    print("Answer:", answer)

    break


Input shape: torch.Size([1, 36])
Answer shape: torch.Size([1, 9])
Input: tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  8, 15, 16,  2,
         17,  8, 18, 19,  3, 20, 21, 22, 23, 24, 13, 28, 26,  1, 29, 30,  8,  6]])
Answer: tensor([[11, 12, 13, 14,  8, 15, 16,  2, 17]])


## 8. Simple RNN Architecture

Architecture:

**Token IDs → Embedding → RNN → Linear → Vocabulary scores**

- Embedding: word ID → dense vector
- RNN: sequence information process করে
- Linear: vocabulary-এর প্রতিটি word-এর score দেয়


In [ ]:
import torch.nn as nn

class SimpleRNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=50,
        hidden_size=64
    ):
        super().__init__()

        # Word IDs → embedding vectors
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Sequence processing
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_size,
            batch_first=True
        )

        # Hidden state → vocabulary scores
        self.fc = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, question):


        embedded = self.embedding(question)

        _, final = self.rnn(embedded)


        output = self.fc(final.squeeze(0))

        return output


model = SimpleRNN(
    vocab_size=len(vocab)
)

print(model)


SimpleRNN(
  (embedding): Embedding(159, 50)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=159, bias=True)
)


## 10. Training

Training steps:

`zero_grad → forward → target → loss → backward → optimizer.step`

এই simple model-এ answer-এর **প্রথম word** target হিসেবে নেওয়া হচ্ছে।


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

epochs = 50

for epoch in range(epochs):

    total_loss = 0

    for question, answer in dataloader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        output = model(question)

        # Only first answer token
        # answer shape: (1, answer_length)
        target = answer[0][0].unsqueeze(0)

        # Calculate loss
        loss = criterion(
            output,
            target
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1:3d}/{epochs} "
            f"| Loss: {total_loss:.4f}"
        )


Epoch  10/50 | Loss: 96.9153
Epoch  20/50 | Loss: 84.9563
Epoch  30/50 | Loss: 70.9288
Epoch  40/50 | Loss: 56.3805
Epoch  50/50 | Loss: 44.9532


## 11. Create Reverse Vocabulary




In [ ]:
idx_to_word = {
    idx: word
    for word, idx in vocab.items()
}

print(list(idx_to_word.items())[:20])


[(0, '<UNK>'), (1, 'mina'), (2, 'found'), (3, 'a'), (4, 'small'), (5, 'blue'), (6, 'umbrella'), (7, 'near'), (8, 'the'), (9, 'school'), (10, 'gate'), (11, 'she'), (12, 'took'), (13, 'it'), (14, 'to'), (15, 'lost'), (16, 'and'), (17, 'room'), (18, 'next'), (19, 'day')]


## 12. Prediction Function

Question → tokenize → indices → model → highest score-এর index → word


In [ ]:
def predict(
    question_text,
    model,
    vocab,
    idx_to_word
):

    model.eval()

    with torch.no_grad():

        # Text → vocabulary indices
        indices = text_to_indices(
            question_text,
            vocab
        )

        if not indices:
            return "<UNK>"

        # Add batch dimension
        x = torch.tensor(
            indices,
            dtype=torch.long
        ).unsqueeze(0)

        # Model prediction
        output = model(x)

        # Highest scoring vocabulary index
        pred = torch.argmax(
            output,
            dim=1
        ).item()

    return idx_to_word.get(
        pred,
        "<UNK>"
    )


## 13. Test the Model

কিছু known question দিয়ে model-এর predicted first answer word দেখছি।


In [ ]:
tests = [
    "Where did Mina find the umbrella?",
    "What did Mina do with the umbrella?",
    "How many tomato seeds did Rafi plant?",
    "Where did Rafi place the pot?",
    "Who helped Lima?",
    "What book did Tuhin borrow?",
    "Where did Sara forget her lunch box?"
]

for q in tests:

    prediction = predict(
        q,
        model,
        vocab,
        idx_to_word
    )

    print(f"Question: {q}")
    print(f"Predicted first answer word: {prediction}")
    print("-" * 70)


Question: Where did Mina find the umbrella?
Predicted first answer word: near
----------------------------------------------------------------------
Question: What did Mina do with the umbrella?
Predicted first answer word: she
----------------------------------------------------------------------
Question: How many tomato seeds did Rafi plant?
Predicted first answer word: three
----------------------------------------------------------------------
Question: Where did Rafi place the pot?
Predicted first answer word: beside
----------------------------------------------------------------------
Question: Who helped Lima?
Predicted first answer word: her
----------------------------------------------------------------------
Question: What book did Tuhin borrow?
Predicted first answer word: a
----------------------------------------------------------------------
Question: Where did Sara forget her lunch box?
Predicted first answer word: during
----------------------------------------------